In [ ]:
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from scipy.special import erf
from scipy.optimize import least_squares
from scipy.stats import genlogistic
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

In [24]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [25]:
# Reused from GammaMixture.ipynb / [MDN] theta to gamma coeffs.ipynb -- keep in
# sync if those notebooks' loader changes.
def load_all_trial_bkt_data(mat_path, ps_indices=None):
    param_sets = {}
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        indices = ps_indices if ps_indices is not None else range(atbd.shape[1])
        for i in indices:
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                trial_data.append(arr.flatten() if arr.dtype == np.float64 else np.array([], dtype=np.float64))
            param_sets[i] = {'p0': p0, 'trial_data': trial_data}
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 2.1*(1 - w[0])           # [1, 2.1]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.3*w[2] + 0.5*(1 - w[2])         # [0.3, 0.5]
    return disc, lr, b


def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial from its raw particle
    arrival times (order statistic), or None if undefined. Copied from
    [per p] distrib of pth percentile bkt.ipynb."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None

In [26]:
# Same connectivity filter as [MDN] theta to gamma coeffs.ipynb.
P0_THRESHOLD = 0.9
Np = 1000

with h5py.File(mat_path, 'r') as f:
    n_param_sets = f['all_trial_bkt_data'].shape[1]

all_param_sets, weights = load_all_trial_bkt_data(mat_path, range(n_param_sets))
USABLE_PS = sorted(i for i, ps in all_param_sets.items() if ps['p0'] < P0_THRESHOLD)
print(f"{len(USABLE_PS)} / {n_param_sets} param sets usable (p0 < {P0_THRESHOLD})")

87 / 120 param sets usable (p0 < 0.9)


In [ ]:
# Kept from the sibling mixture notebooks as motivating context (not used by
# this notebook's fit itself, which doesn't use a Gaussian mixture at all):
# is a 2-component Gaussian mixture (in log space) actually sufficient for the
# distribution of pth-percentile BKT across trials, at fixed (theta,p)? Check
# via BIC across K=1,2,3 for a spread of param sets (by connectivity) and
# percentiles, rather than assuming. The supervisor's own read of the real
# empirical CDFs (not clearly showing 2 Gaussians) is the direct motivation
# for trying the single skew-logistic S-curve below instead.
CHECK_P_VALS = [0.01, 0.1, 0.5, 0.9]
ps_by_p0 = sorted(USABLE_PS, key=lambda i: all_param_sets[i]['p0'])
CHECK_PS = [ps_by_p0[0], ps_by_p0[len(ps_by_p0) // 2], ps_by_p0[-1]]  # well-connected / mid / marginal

fig, axes = plt.subplots(len(CHECK_PS), len(CHECK_P_VALS), figsize=(4 * len(CHECK_P_VALS), 3.2 * len(CHECK_PS)))
for row, ps_idx in enumerate(CHECK_PS):
    ps = all_param_sets[ps_idx]
    for col, p in enumerate(CHECK_P_VALS):
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        ax = axes[row, col]
        if len(vals) < 10:
            ax.set_title(f'ps{ps_idx} p={p} (n={len(vals)}, too few)')
            continue
        logvals = np.log(vals).reshape(-1, 1)
        bics = {k: GaussianMixture(n_components=k, random_state=0, n_init=3).fit(logvals).bic(logvals)
                for k in [1, 2, 3]}
        best_k = min(bics, key=bics.get)
        ax.hist(logvals, bins=40, density=True, alpha=0.5)
        xg = np.linspace(logvals.min(), logvals.max(), 300).reshape(-1, 1)
        for k in [1, 2, 3]:
            gm = GaussianMixture(n_components=k, random_state=0, n_init=3).fit(logvals)
            ax.plot(xg, np.exp(gm.score_samples(xg)), label=f'K={k} BIC={bics[k]:.0f}',
                     lw=1.5 if k == best_k else 1, ls='-' if k == best_k else '--')
        ax.set_title(f'ps{ps_idx} (p0={ps["p0"]:.2f}) p={p}\nn={len(vals)}, best K={best_k}')
        ax.legend(fontsize=6)
plt.tight_layout()
plt.savefig(transport_dir / 'percentile_gmm_K_check.png', dpi=110)
plt.show()

# Finding (see conclusion cell): 2 components is a reasonable default -- matches
# GMM.ipynb's precedent, and clearly beats K=1 for well-connected/high-sample
# param sets (skewed, long-tailed shape). But it's not obviously "correct":
# K=3 keeps improving BIC further for high-n groups (mild residual tail
# structure), while low-sample/marginal-connectivity groups actually PREFER
# K=1 (2 components risks fitting noise there). We proceed with K=2 as
# requested, but this is the evidence behind that choice, not an assumption.
# K is not used below -- this notebook fits a single 3-parameter skew-logistic
# curve per (theta,p) group, not a K-component mixture.

In [ ]:
# Precompute stage: fit each (theta,p) group's skew-logistic CDF parameters
# (mu, s, nu) via least-squares against the empirical CDF -- the third arm of
# the 3-way model comparison (see the PDF/MLE baseline notebook's conclusion
# cell). This is the Type I Generalized Logistic Distribution (a.k.a.
# skew-logistic / Richards curve):
#   F(x; mu, s, nu) = 1 / (1 + exp(-(x-mu)/s))^nu,   x = log10(t)
# nu=1 recovers the ordinary symmetric logistic; nu != 1 gives asymmetry
# (steeper approach to 0 than to 1, or vice versa) -- exactly the "skewed
# S-curve" shape the supervisor suggested after noting the real empirical
# CDFs don't clearly show a 2-Gaussian mixture. Unlike the mixture notebooks,
# this is a single smooth monotonic function with only 3 parameters, so there
# is no "weakly-identified minor component" problem at all -- no
# responsibility weighting needed downstream (cell-7).
#
# scipy.stats.genlogistic implements exactly this family (its shape param `c`
# is our `nu`), which gives a free MLE cross-check for validating the custom
# least-squares fit -- see fit_genlogistic_cdf's docstring and the validation
# script referenced there. On a synthetic case (true mu=5.2, s=0.35, nu=2.5,
# N=2000), the custom LS fit's resulting CDF matched scipy's own MLE fit
# CDF closely across the range (e.g. F(6.14): custom=0.842, scipy_mle=0.844,
# true=0.850) -- point estimates of (mu,s,nu) differ slightly since the three
# parameters are mildly correlated/not perfectly separable at finite N, but
# what matters for this notebook (the resulting CDF shape) checked out.
#
# Same p-grid as the mixture notebooks (p is free to sample densely, no new
# simulation needed).
p_vals = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3,
                    0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
MIN_SAMPLES = 20


def genlogistic_cdf(x, mu, s, nu):
    """Type I generalized logistic (skew-logistic) CDF, x=log10(t). Already
    valid on [0,1] by construction (a sigmoid raised to a positive power),
    no erf-style shift/rescale needed. Computed as exp(-nu*softplus(-z)) via
    np.logaddexp rather than 1/(1+exp(-z))**nu directly -- the direct form
    overflows (exp(-z) or its power blows up) for the large-nu, large-|z|
    combinations that turn out to be common in this dataset (see the note in
    fit_genlogistic_cdf below); this form is mathematically identical but
    numerically stable for any nu, z."""
    z = (x - mu) / s
    return np.exp(-nu * np.logaddexp(0.0, -z))


def fit_genlogistic_cdf(vals, n_init=8, seed=0, nu_hi=20.0):
    """Fit (mu, s, nu) by least-squares against the empirical CDF (order
    statistics), mirroring fit_erf_mixture_cdf's structure in the sibling
    2-erf-mixture notebook: mu is bounded to a generous-but-finite margin
    around the group's own data range (same rationale as that notebook --
    prevents a poorly-constrained parameter from drifting to an extreme,
    near-free-SSE-reduction value), s>0 bounded similarly.

    nu is bounded to (1e-3, nu_hi=20) -- NOT because 20 is generous enough to
    rarely bind (it isn't: ~42% of the 1479 real fits below hit this bound),
    but because raising it doesn't help. Probed nu_hi in {20,100,1000} on a
    15-param-set sample: the at-bound fraction barely drops (23%->20%->17%)
    and sse barely improves (mean 0.061->0.058->0.057) as the bound is
    raised 50x -- nu keeps growing without meaningfully improving the fit,
    which is the signature of an asymptotic limit, not an under-tight bound.
    In other words: for a real, non-trivial fraction of (theta,p) groups, a
    single skew-logistic curve structurally cannot represent the empirical
    CDF's shape well no matter how skewed (nu->inf approaches a step
    function) -- direct evidence supporting the mixture notebooks' 2-Gaussian
    structure over a single S-curve for at least those groups. See the
    conclusion cell for the full comparison."""
    x = np.sort(np.log10(np.asarray(vals, dtype=float)))
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 5:
        return None
    ecdf = (np.arange(1, n + 1) - 0.5) / n  # standard plotting-position ECDF estimator

    data_lo, data_hi = x[0], x[-1]
    data_range = max(data_hi - data_lo, 1e-3)
    mu_lo, mu_hi = data_lo - 3 * data_range, data_hi + 3 * data_range
    s_hi = 5 * data_range

    rng = np.random.default_rng(seed)

    def residuals(params):
        mu, s, nu = params
        return genlogistic_cdf(x, mu, s, nu) - ecdf

    best = None
    for _ in range(n_init):
        mu0 = np.median(x) + rng.uniform(-0.3, 0.3) * data_range
        s0 = max(x.std(), 1e-3) * rng.uniform(0.5, 1.5)
        nu0 = rng.uniform(0.3, 3.0)
        x0 = [mu0, s0, nu0]
        try:
            res = least_squares(residuals, x0,
                                 bounds=([mu_lo, 1e-6, 1e-3], [mu_hi, s_hi, nu_hi]))
        except Exception:
            continue
        sse = np.sum(res.fun ** 2)
        if best is None or sse < best[0]:
            best = (sse, res.x)

    if best is None:
        return None
    sse, (mu, s, nu) = best
    return {'mu': mu, 's': s, 'nu': nu, 'sse': sse}


records = []
for ps_idx in USABLE_PS:
    ps = all_param_sets[ps_idx]
    disc, lr, b = recover_theta(weights[ps_idx])
    for p in p_vals:
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        if len(vals) < MIN_SAMPLES:
            continue
        fit = fit_genlogistic_cdf(vals)
        if fit is None:
            continue
        records.append({
            'paramset_idx': ps_idx, 'discspacing': disc, 'length_ratio': lr, 'b': b, 'p': p,
            'n_samples': len(vals),
            **fit,
        })

fit_df = pd.DataFrame(records)
print(f"Fit {len(fit_df)} (theta,p) groups across {fit_df['paramset_idx'].nunique()} param sets")
print(fit_df['sse'].describe())
print("\nnu (skew) distribution -- nu=1 is symmetric logistic:")
print(fit_df['nu'].describe())
n_at_bound = (fit_df['nu'] >= 19.99).sum()
print(f"\nnu at upper bound (20): {n_at_bound} / {len(fit_df)} ({100*n_at_bound/len(fit_df):.1f}%) -- see fit_genlogistic_cdf's docstring")


In [ ]:
# Illustration: what does the active skew-logistic CDF fit (used in the
# precompute stage above) look like against the real empirical CDF? Same two
# representative (param set, p) examples as the sibling notebooks, titled by
# physical parameters (d=discspacing, l=length_ratio, b=aperture).
EXAMPLE_CASES = [(ps_by_p0[len(ps_by_p0) // 2], 0.5), (ps_by_p0[0], 0.1)]

fig, axes = plt.subplots(1, len(EXAMPLE_CASES), figsize=(6.5 * len(EXAMPLE_CASES), 4.5))
for ax, (ps_idx, p) in zip(np.atleast_1d(axes), EXAMPLE_CASES):
    ps = all_param_sets[ps_idx]
    d, l, b = recover_theta(weights[ps_idx])
    vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
    vals = np.array([v for v in vals if v is not None])
    logvals = np.sort(np.log10(vals))
    n = len(logvals)
    ecdf = (np.arange(1, n + 1) - 0.5) / n

    fit = fit_genlogistic_cdf(vals)

    ax.plot(logvals, ecdf, 'o', ms=3, alpha=0.5, label='empirical CDF')
    xg = np.linspace(logvals.min() - 0.2, logvals.max() + 0.2, 300)
    pred_cdf = genlogistic_cdf(xg, fit['mu'], fit['s'], fit['nu'])
    ax.plot(xg, pred_cdf, 'g-', lw=2, label='skew-logistic CDF fit')

    ticks = ax.get_xticks()
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
    ax.set_xlabel('breakthrough time (s)')
    ax.set_ylabel('CDF')
    ax.set_title(f'd={d:.2f}, l={l:.2f}, b={b:.2e}\np={p}, n={len(vals)}, nu={fit["nu"]:.2f}, sse={fit["sse"]:.4f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(transport_dir / 'percentile_skewlogistic_examples.png', dpi=120)
plt.show()


In [ ]:
# Build the regression dataset: input (discspacing, length_ratio, b, logit(p)),
# target (mu, s, nu), both z-scored. logit(p) rather than raw p since p is a
# probability. Same unconstrained-output-transform pattern as the mixture
# notebooks (train on logit/log of bounded parameters, only invert at
# denormalization time) -- here only s and nu need it (mu is already
# unconstrained/real-valued).
FEATURE_COLS = ['discspacing', 'length_ratio', 'b']
TARGET_COLS = ['mu', 's', 'nu']  # real, constrained parameter space (s>0, nu>0)


def logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def transform_Y(Y):
    """Real parameter space -> unconstrained space the network is trained on."""
    mu, s, nu = Y[:, 0], Y[:, 1], Y[:, 2]
    return np.column_stack([mu, np.log(s), np.log(nu)])


def inverse_transform_Y(Yt):
    """Unconstrained space -> real parameter space (guarantees s>0, nu>0)."""
    mu, ls, lnu = Yt[:, 0], Yt[:, 1], Yt[:, 2]
    return np.column_stack([mu, np.exp(ls), np.exp(lnu)])


X_all = np.column_stack([fit_df[FEATURE_COLS].values, logit(fit_df['p'].values)])
Y_all = transform_Y(fit_df[TARGET_COLS].values)  # train on the unconstrained transform

X_mean, X_std = X_all.mean(axis=0), X_all.std(axis=0)
Y_mean, Y_std = Y_all.mean(axis=0), Y_all.std(axis=0)


def normalize_X(X):
    return (X - X_mean) / X_std


def normalize_Y(Y):
    return (Y - Y_mean) / Y_std


def denormalize_Y(Y):
    """Network output (normalized, unconstrained space) -> real parameter
    space. Composes the z-score inverse with inverse_transform_Y so every
    call-site gets valid parameters automatically."""
    return inverse_transform_Y(Y * Y_std + Y_mean)

In [ ]:
# Regression network -- same architecture/training recipe as the mixture
# notebooks (Dense 4->20->20->20->20->N, L2=9e-5, Adam with a stepped LR
# schedule), just with the output narrowed from 5 to 3 (mu, s, nu instead of
# w1,mu1,sigma1,mu2,sigma2).
#
# Loss: plain MSE on the normalized+transformed targets -- no responsibility
# weighting needed here. That machinery existed in the mixture notebooks
# specifically because a low-weight mixture component's (mu,sigma) are
# themselves poorly identified by the data (noisy ground truth the network
# shouldn't be forced to chase equally hard). A single skew-logistic curve
# has no such "irrelevant component" -- all 3 parameters always describe the
# one and only curve being fit, so there's nothing analogous to down-weight.
def build_model():
    l2_reg = 9e-5
    return keras.Sequential([
        keras.layers.Input(shape=(4,)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(3),
    ])


def lr_by_epoch(epoch):
    if epoch < 200:
        return 1e-3
    elif epoch < 300:
        return 5e-4
    else:
        return 1e-4


def train_model(X_train, Y_train, X_val, Y_val, epochs=400, verbose=0):
    model = build_model()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs,
                         batch_size=32, callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch, verbose=0)],
                         verbose=verbose)
    return model, history

In [ ]:
# Train the main model on a random 80/20 split (matches GMM.ipynb's convention).
X_norm, Y_norm = normalize_X(X_all), normalize_Y(Y_all)
X_train, X_val, Y_train, Y_val = train_test_split(X_norm, Y_norm, test_size=0.2, random_state=42)

model, history = train_model(X_train, Y_train, X_val, Y_val, epochs=400, verbose=0)
print(f"final train loss: {history.history['loss'][-1]:.4f}  val loss: {history.history['val_loss'][-1]:.4f}")
model.save(transport_dir / 'percentile_cdf_skewlogistic_model.keras')

In [ ]:
# --- Experimental: monotonicity-across-p penalized training ---
#
# Physical requirement not enforced anywhere above: for a FIXED theta, if
# p1 < p2, the p1 CDF curve must lie entirely at-or-left-of (>= in CDF value
# at any fixed x, essentially) the p2 curve -- percentile-BKT is literally an
# order statistic of the same sorted per-trial array, so predicted CDFs for
# different p must not cross. Independent per-(theta,p) regression (cell-8's
# `model`) has no mechanism to know about this relationship between rows.
#
# This is an EXPLORATORY attempt to enforce it during training, framed by the
# supervisor as "an interesting constraint to try," not a guaranteed fix --
# reported here with an honest before/after comparison, not an assumed win.
#
# Approach: group rows by param set (all p_vals for one theta together,
# required so predictions across different p can be compared) and add a
# hinge-style penalty for each ADJACENT pair p_i < p_{i+1} within a group:
#   mean(relu(CDF_{p_{i+1}}(x_grid) - CDF_{p_i}(x_grid)))
# (penalizes CDF_{p_i} < CDF_{p_{i+1}}, since the correct relationship for
# p_i < p_{i+1} is CDF_{p_i}(x) >= CDF_{p_{i+1}}(x) everywhere -- verified
# directly against real data: e.g. param set 119, p=0.01 vs p=0.5, at the
# median log-time CDF_0.01=0.855 >> CDF_0.5=0.145, confirming smaller-p
# quantiles are stochastically smaller values and their CDF rises to 1
# sooner. NOTE: an earlier version of this cell had this backwards --
# `cdfs[:, :-1] - cdfs[:, 1:]`, i.e. CDF_i - CDF_{i+1}, penalized whenever
# that was POSITIVE, which is actually the physically correct configuration
# -- so it was pushing the network AWAY from the real constraint. Fixed
# below; see mono-conclusion for corrected before/after numbers.)
# over a per-group x_grid spanning that param set's own fitted mu +/- 4*s
# range. Only adjacent pairs are needed (not all pairs) since p_vals is
# already sorted ascending -- monotonicity of adjacent pairs implies it for
# all pairs by transitivity. Needs a custom tf.GradientTape training loop
# rather than plain model.fit(), since standard row-shuffled batching
# doesn't group rows by theta -- which forces a second change described next.
#
# Batching by param set also forces the train/val SPLIT to be by param set
# (a param set's rows can't be split across train/val, or there's nothing to
# compare within a val batch either) -- unlike cell-8, whose row-random split
# leaves nearly every param set with SOME rows in both train and val. That
# difference turned out to matter far more than the batching mechanics:
#
# - FIRST ATTEMPT: one gradient step per param set (batch size ~17, all
#   sharing one theta). Held-out MSE looked disastrous (0.30 vs cell-8's
#   0.10) -- but so did a lam=0 ablation with identical batching/split (0.29),
#   proving the penalty wasn't the cause.
# - Tried META_SIZE=8 (pool 8 param sets/step) and META_SIZE=2: NEITHER
#   fixed it (val loss 0.30-0.34 either way, no clear trend with batch size).
#   That ruled out "too few gradient steps per epoch" as the explanation too.
# - Actual cause: this is a genuinely HARDER generalization test than
#   cell-8's. Held-out param sets here are THETAS NEVER SEEN IN TRAINING AT
#   ALL (extrapolation across theta-space); cell-8's "held-out" rows mostly
#   share their theta with other rows already in its training set (just
#   different p) -- an easier interpolation-across-p task for an
#   already-familiar theta. The two val losses are answering different
#   questions and were never comparable in the first place.
#
# So the right "before/after" comparison is apples-to-apples: SAME by-
# param-set split, SAME meta-batching, WITH vs WITHOUT the penalty (lam=1 vs
# lam=0) -- both cells below.
META_SIZE = 8

groups = []
for ps_idx, g in fit_df.groupby('paramset_idx'):
    g = g.sort_values('p')
    if len(g) < 2:
        continue  # no adjacent pair to penalize
    X_g = normalize_X(np.column_stack([g[FEATURE_COLS].values, logit(g['p'].values)])).astype(np.float32)
    Y_g = normalize_Y(transform_Y(g[TARGET_COLS].values)).astype(np.float32)
    x_lo = (g['mu'] - 4 * g['s']).min()
    x_hi = (g['mu'] + 4 * g['s']).max()
    grid_g = np.linspace(x_lo, x_hi, 40).astype(np.float32)
    groups.append((X_g, Y_g, grid_g))

print(f"{len(groups)} param-set groups with >=2 p_vals (out of {fit_df['paramset_idx'].nunique()} total)")

_Y_mean_tf = tf.constant(Y_mean, dtype=tf.float32)
_Y_std_tf = tf.constant(Y_std, dtype=tf.float32)


def denormalize_Y_tf(Yn):
    """TF-differentiable version of denormalize_Y (normalized+transformed
    network output -> real mu,s,nu), needed so the monotonicity penalty's
    gradient can flow back into the network weights."""
    Yt = Yn * _Y_std_tf + _Y_mean_tf
    mu, ls, lnu = Yt[:, 0], Yt[:, 1], Yt[:, 2]
    return mu, tf.exp(ls), tf.exp(lnu)


def genlogistic_cdf_tf(x, mu, s, nu):
    """TF-differentiable version of genlogistic_cdf, same numerically-stable
    softplus form as the numpy version in cell-5."""
    z = (x[:, None] - mu[None, :]) / s[None, :]  # (grid, n_p)
    return tf.exp(-nu[None, :] * tf.math.softplus(-z))


def monotonicity_penalty_one_group(Y_pred_norm_slice, grid):
    """Y_pred_norm_slice: (n_p, 3) network outputs for ONE param set's rows
    (a slice of the meta-batch), sorted ascending by p. Returns scalar hinge
    penalty over adjacent p pairs for that param set alone.

    Correct relationship for p_i < p_{i+1}: CDF_{p_i}(x) >= CDF_{p_{i+1}}(x)
    everywhere (smaller-percentile quantiles are stochastically smaller
    values, so that curve reaches 1 sooner). A violation is therefore
    CDF_{p_i} < CDF_{p_{i+1}}, i.e. a positive (CDF_{p_{i+1}} - CDF_{p_i})."""
    mu, s, nu = denormalize_Y_tf(Y_pred_norm_slice)
    cdfs = genlogistic_cdf_tf(tf.constant(grid), mu, s, nu)  # (n_grid, n_p)
    violation = cdfs[:, 1:] - cdfs[:, :-1]  # CDF_{i+1} - CDF_i, should be <=0
    return tf.reduce_mean(tf.nn.relu(violation))


def make_meta_batches(groups, meta_size, rng):
    order = rng.permutation(len(groups))
    for i in range(0, len(order), meta_size):
        yield [groups[j] for j in order[i:i + meta_size]]


def train_model_monotonic(groups, val_groups, epochs=400, lam=1.0, meta_size=META_SIZE, verbose=0):
    model = build_model()
    opt = keras.optimizers.Adam(1e-3)
    rng = np.random.default_rng(0)
    history = {'loss': [], 'val_loss': [], 'mono_penalty': []}

    for epoch in range(epochs):
        opt.learning_rate.assign(lr_by_epoch(epoch))
        epoch_loss, epoch_mono = [], []
        for meta_batch in make_meta_batches(groups, meta_size, rng):
            X_cat = np.concatenate([X_g for X_g, _, _ in meta_batch], axis=0)
            Y_cat = np.concatenate([Y_g for _, Y_g, _ in meta_batch], axis=0)
            sizes = [len(X_g) for X_g, _, _ in meta_batch]
            with tf.GradientTape() as tape:
                Y_pred = model(X_cat, training=True)
                reg_loss = tf.reduce_mean(tf.square(Y_cat - Y_pred))  # pooled over the WHOLE meta-batch
                offset = 0
                mono_losses = []
                for (_, _, grid_g), n_g in zip(meta_batch, sizes):
                    mono_losses.append(monotonicity_penalty_one_group(Y_pred[offset:offset + n_g], grid_g))
                    offset += n_g
                mono_loss = tf.reduce_mean(tf.stack(mono_losses))  # averaged across param sets, not rows
                loss = reg_loss + lam * mono_loss
            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))
            epoch_loss.append(float(reg_loss))
            epoch_mono.append(float(mono_loss))

        val_losses = []
        for X_g, Y_g, grid_g in val_groups:
            Y_pred = model(X_g, training=False)
            val_losses.append(float(tf.reduce_mean(tf.square(Y_g - Y_pred))))

        history['loss'].append(np.mean(epoch_loss))
        history['mono_penalty'].append(np.mean(epoch_mono))
        history['val_loss'].append(np.mean(val_losses) if val_losses else np.nan)
        if verbose and epoch % 50 == 0:
            print(f"epoch {epoch}: loss={history['loss'][-1]:.4f} "
                  f"mono={history['mono_penalty'][-1]:.4f} val_loss={history['val_loss'][-1]:.4f}")

    return model, history


# Same 80/20 split, but split by PARAM SET (not row) so a param set's rows
# stay together in either train or val.
rng = np.random.default_rng(42)
n_val_ps = max(1, int(0.2 * len(groups)))
perm = rng.permutation(len(groups))
val_idx, train_idx = perm[:n_val_ps], perm[n_val_ps:]
train_groups = [groups[i] for i in train_idx]
val_groups = [groups[i] for i in val_idx]

model_mono, history_mono = train_model_monotonic(train_groups, val_groups, epochs=400, lam=1.0, verbose=0)
print(f"WITH penalty (lam=1): final train loss: {history_mono['loss'][-1]:.4f}  "
      f"val loss: {history_mono['val_loss'][-1]:.4f}  mono penalty: {history_mono['mono_penalty'][-1]:.5f}")
model_mono.save(transport_dir / 'percentile_cdf_skewlogistic_model_monotonic.keras')

# Fair apples-to-apples control: IDENTICAL split and meta-batching, but
# lam=0 (no monotonicity penalty) -- isolates what the penalty itself costs,
# separate from the harder (novel-theta) train/val split used here.
model_nopenalty, history_nopenalty = train_model_monotonic(train_groups, val_groups, epochs=400, lam=0.0, verbose=0)
print(f"WITHOUT penalty (lam=0, same split/batching): final train loss: {history_nopenalty['loss'][-1]:.4f}  "
      f"val loss: {history_nopenalty['val_loss'][-1]:.4f}")


In [ ]:
# Before/after check: does the monotonicity penalty actually reduce
# curve-crossing, and at what cost to fit quality? Informal check per the
# plan (not a full separate diagnostic deliverable).
#
# Two comparisons, for two different questions:
# 1. violation rate: model (cell-8, plain regression) vs model_mono
#    (monotonicity-penalized) -- both evaluated on ALL usable param sets
#    (fair here, since violation-checking doesn't care which param sets each
#    model trained on).
# 2. fit-quality cost of the penalty: model_nopenalty vs model_mono -- the
#    FAIR comparison (identical split/batching, only lam differs), not
#    model (cell-8) vs model_mono (which differ in split AND batching, per
#    mono-train's header comment).
def count_violations(model_to_check, ps_indices, p_vals_check):
    """A violation is CDF_{p_i}(x) < CDF_{p_{i+1}}(x) for some x, when
    p_i < p_{i+1} -- i.e. diff = CDF_i - CDF_{i+1} should be >= 0 everywhere
    (see mono-train's monotonicity_penalty_one_group docstring for the
    physical justification/verification). Previously this checked
    `diff.max() > 1e-6` (flagging when diff was POSITIVE), which is backwards
    -- that's the physically correct configuration, not a violation. Fixed
    to flag `diff.min() < -1e-6` instead."""
    total_pairs, violating_pairs, max_violation = 0, 0, 0.0
    for ps_idx in ps_indices:
        disc, lr, b = recover_theta(weights[ps_idx])
        X_ps = normalize_X(np.array([[disc, lr, b, logit(p)] for p in p_vals_check]))
        mu, s, nu = denormalize_Y(model_to_check.predict(X_ps, verbose=0)).T
        x_lo, x_hi = (mu - 4 * s).min(), (mu + 4 * s).max()
        xg = np.linspace(x_lo, x_hi, 80)
        cdfs = np.stack([genlogistic_cdf(xg, mu[i], s[i], nu[i]) for i in range(len(p_vals_check))], axis=1)
        for i in range(len(p_vals_check) - 1):
            total_pairs += 1
            diff = cdfs[:, i] - cdfs[:, i + 1]  # should be >= 0 everywhere (CDF_i >= CDF_{i+1})
            if diff.min() < -1e-6:
                violating_pairs += 1
                max_violation = max(max_violation, -diff.min())
    return total_pairs, violating_pairs, max_violation


check_p_vals = [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]  # same grid as cell-10's plot_p_vals
tot_before, viol_before, max_before = count_violations(model, USABLE_PS, check_p_vals)
tot_after, viol_after, max_after = count_violations(model_mono, USABLE_PS, check_p_vals)
tot_nopen, viol_nopen, max_nopen = count_violations(model_nopenalty, USABLE_PS, check_p_vals)

print("Violation rate across ALL usable param sets:")
print(f"  cell-8 model (plain regression, row-random split):        {viol_before}/{tot_before} "
      f"({100*viol_before/tot_before:.1f}%), max={max_before:.4f}")
print(f"  model_nopenalty (by-paramset split, lam=0, same batching): {viol_nopen}/{tot_nopen} "
      f"({100*viol_nopen/tot_nopen:.1f}%), max={max_nopen:.4f}")
print(f"  model_mono (by-paramset split, lam=1, same batching):      {viol_after}/{tot_after} "
      f"({100*viol_after/tot_after:.1f}%), max={max_after:.4f}")

# Fair fit-quality trade-off: same split, same batching, only lam differs.
val_mse_nopen, val_mse_mono = [], []
for X_g, Y_g, grid_g in val_groups:
    val_mse_nopen.append(np.mean((Y_g - model_nopenalty.predict(X_g, verbose=0)) ** 2))
    val_mse_mono.append(np.mean((Y_g - model_mono.predict(X_g, verbose=0)) ** 2))
print(f"\nheld-out regression MSE, SAME split/batching (fair comparison, penalty term excluded):")
print(f"  lam=0 (no penalty): {np.mean(val_mse_nopen):.4f}   lam=1 (penalty): {np.mean(val_mse_mono):.4f}")

In [ ]:
# For every usable param set, plot the empirical pth-percentile-BKT CDF
# (solid, color-coded by p) with the NN-predicted skew-logistic CDF for that
# (theta,p) overlaid in the same color but dashed -- one page per param set
# in a multi-page PDF. Uses the cell-8 `model` (trained on the full dataset)
# as a full-dataset visual QC pass.
plot_p_vals = [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]  # same as sibling notebooks
plot_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(plot_p_vals)))

plot_rows = [(ps_idx, p) for ps_idx in USABLE_PS for p in plot_p_vals]
X_plot = np.array([[*recover_theta(weights[ps_idx]), logit(p)] for ps_idx, p in plot_rows])
Y_plot_pred = denormalize_Y(model.predict(normalize_X(X_plot), verbose=0))
pred_lookup = {key: Y_plot_pred[i] for i, key in enumerate(plot_rows)}

pdf_path = transport_dir / '[per p] distrib of pth percentile bkt, empirical vs NN (CDF fit, skew-logistic).pdf'
with PdfPages(pdf_path) as pdf:
    for ps_idx in USABLE_PS:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        fig, ax = plt.subplots(figsize=(9, 6))
        for p, color in zip(plot_p_vals, plot_colors):
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            if len(vals) < 2:
                continue

            log_vals = np.sort(np.log10(vals))
            n = len(log_vals)
            ecdf = (np.arange(1, n + 1) - 0.5) / n
            ax.plot(log_vals, ecdf, color=color, linestyle='-', label=f'p={p:.3f} (n={n})')

            mu, s, nu = pred_lookup[(ps_idx, p)]
            xg = np.linspace(log_vals.min() - 0.3, log_vals.max() + 0.3, 300)
            pred_cdf = genlogistic_cdf(xg, mu, s, nu)
            ax.plot(xg, pred_cdf, color=color, linestyle='--')

        ax.set_xlabel('breakthrough time (s)')
        ax.set_ylabel('CDF')
        ticks = ax.get_xticks()
        ax.set_xticks(ticks)
        ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax.set_ylim(-0.02, 1.02)
        ax.set_title(f'Param set {ps_idx}  disc={disc:.2f}, lr={lr:.2f}, b={b:.2e}, p0={ps["p0"]:.2f}\n'
                     f'solid = empirical CDF, dashed = NN-predicted skew-logistic CDF')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}  ({len(USABLE_PS)} pages)')

In [ ]:
# --- Conclusion ---
#
# This is the **CDF / skew-logistic** notebook of the 3-way model comparison
# (see the PDF/MLE baseline's conclusion cell in
# "[per p] distrib of pth percentile bkt GMM coeffs.ipynb" for the full
# framing). Single 3-parameter Type I generalized logistic CDF per (theta,p),
# fit by least-squares against the empirical CDF (cell-5's
# fit_genlogistic_cdf, validated against scipy.stats.genlogistic's MLE on a
# synthetic case -- see cell-5's header comment). No mixture, so no
# responsibility-weighted loss needed (cell-7) -- plain MSE on the
# unconstrained-transformed targets.
#
# Precompute (cell-5), all three notebooks' sse (CDF-space, lower=better fit):
#   PDF/MLE baseline (2-Gaussian mixture, via density MLE):      mean 0.1015
#   CDF / 2-erf mixture (2-Gaussian mixture, via CDF LS):        mean 0.0386
#   CDF / skew-logistic (this notebook, single S-curve, via LS): mean 0.0928
# This notebook's fit is clearly worse than the 2-erf-mixture notebook's
# (both use direct CDF least-squares, so it's an apples-to-apples comparison
# of the two model FAMILIES, not the fitting method) -- and NOT because the
# `nu` bound of 20 is too tight: 41.5% of the 1479 fits hit that bound, but
# raising it 50x (to 1000) on a probe subsample barely moved sse (see
# fit_genlogistic_cdf's docstring for the numbers). A single S-curve
# genuinely cannot represent a real fraction of these distributions' shapes,
# no matter how skewed. Visually (the example-cases cell above): both
# representative cases show the same signature mismatch -- the fitted curve
# rounds off toward CDF=1 measurably before the empirical data does, i.e. a
# real, visible discrepancy in the tail/shoulder, not just a larger sse
# number in the abstract. This is direct evidence AGAINST the supervisor's
# tentative read that a single skewed S-curve might replace the 2-Gaussian
# mixture -- at least for this dataset, the extra flexibility of 2 components
# (in either the PDF/MLE or CDF-LS form) matters for a meaningful fraction of
# (theta,p) groups.
#
# Regression network (cell-8), train/val loss across all three notebooks:
#   PDF/MLE baseline:    0.0736 / 0.2219  (5 outputs, responsibility-weighted)
#   CDF / 2-erf mixture: 0.1589 / 0.2790  (5 outputs, responsibility-weighted)
#   CDF / skew-logistic: 0.0400 / 0.0764  (3 outputs, plain MSE)
# Lowest loss of the three, plausibly because there's no weakly-identified
# extra dimension to fight (only 3 outputs, no per-example noisy minor-
# component target) -- but these losses are NOT on a shared scale (different
# number of outputs, different targets/units after the unconstrained
# transform), so this is NOT evidence that the skew-logistic model
# "generalizes best" -- it's evidence that ITS OWN regression targets are
# easier to regress onto, which is a different (and less important) property
# than whether those targets describe the data well in the first place (the
# sse comparison above, and cell-10's diagnostic PDF, are the relevant checks
# for that).
#
# Net takeaway: despite easier downstream regression, the single
# skew-logistic curve is the weakest of the three model families at
# representing the actual empirical CDFs for a meaningful fraction of
# (theta,p) groups (~40% hit the nu bound trying and failing to get skewed
# enough). Between the two CDF-fitting families, the 2-erf mixture is the
# better-fitting one; between the two fitting METHODS for that same mixture
# family, direct CDF least-squares clearly beats MLE at the precompute stage
# (0.0386 vs 0.1015 sse) but produces harder-to-regress targets downstream
# (worse train/val loss than the PDF/MLE baseline) -- so there isn't a single
# unambiguous winner among all three without deciding which failure mode
# matters more for the eventual use case (fit fidelity per group vs.
# smoothness/generalization of the theta->coefficients mapping).

In [ ]:
# --- Addendum: monotonicity-penalized training (experimental) ---
#
# CORRECTION (supersedes the numbers originally reported here): the
# violation check and the training penalty both had an inverted sign --
# `monotonicity_penalty_one_group` penalized CDF_{p_i} > CDF_{p_{i+1}}, and
# `count_violations` flagged the same (positive) direction as a violation.
# That is backwards: for p_i < p_{i+1}, the physically correct relationship
# is CDF_{p_i}(x) >= CDF_{p_{i+1}}(x) everywhere (percentile-BKT is an order
# statistic of the same sorted per-trial array -- a smaller-percentile
# quantile is a stochastically SMALLER value, so its CDF reaches 1 sooner).
# Verified directly against real data (param set 119, p=0.01 vs p=0.5: at
# the median log-time, CDF_0.01=0.855 >> CDF_0.5=0.145). So the old code was
# training the penalty AWAY from the real constraint and counting the
# physically-correct configuration as a "violation." Both are fixed now
# (mono-train's monotonicity_penalty_one_group and mono-compare's
# count_violations) -- the numbers below are from the corrected run.
#
# Getting a FAIR before/after comparison took three iterations (see
# mono-train's header comment for the full trail): batching by param set
# forces the train/val split to also be by param set (entire thetas held
# out, never seen in training at all), which is a strictly harder
# generalization test than cell-8's row-random split (where nearly every
# theta appears in both train and val, just under different p -- an easier
# interpolation task). The apples-to-apples control (model_nopenalty:
# IDENTICAL by-paramset split and meta-batching, lam=0) isolates the
# penalty's own effect from that split difference.
#
# Corrected results (all on the SAME held-out param sets, 609 adjacent-p
# pairs across all 87 usable param sets):
#   violation rate:  cell-8 model 37.6%  ->  model_nopenalty 39.2%  (same
#     split/batching, no penalty -- roughly unchanged from cell-8, as
#     expected, modulo the harder split) -> model_mono 30.5% (WITH penalty)
#     -- a real but modest ~9-point reduction, not the ~24-point drop
#     originally (mis)reported.
#   held-out regression MSE (fair comparison): lam=0 0.2676 -> lam=1 0.2703
#     -- the penalty costs a small amount of fit quality (+1%), rather than
#     improving it as the buggy version claimed.
#
# Honest reading: the monotonicity penalty, correctly implemented, does what
# it's supposed to -- it measurably reduces curve-crossing (37.6-39.2% ->
# 30.5%) -- at a small, roughly break-even cost to held-out MSE. It does NOT
# eliminate violations (majority of pairs, 69.5%, still cross on this finer
# 8-value check grid vs. the 17-value grid used for fitting), and this is a
# SINGLE run (one random init, one 80/20 theta split) so the exact magnitude
# shouldn't be over-read. The core exploratory question the supervisor posed
# -- "can this be enforced during training?" -- gets a modest yes: a real,
# non-trivial reduction in violations for a small, defensible cost in fit
# quality, not a dramatic fix.
#
# Caveats on this implementation, for whoever extends it:
# - The penalty is evaluated on a per-param-set x_grid derived from the
#   FITTED (ground-truth) mu/s range for that param set, not the network's
#   own predicted range -- reasonable while predictions are close to the
#   fits, but could under-penalize a badly-off prediction whose true crossing
#   region falls outside that grid.
# - Only ADJACENT p pairs (in the p_vals grid used for fitting) are
#   penalized, relying on transitivity for the full ordering -- correct in
#   theory, but only as good as how finely p_vals is sampled; two adjacent
#   p's that are far apart could still have crossing predicted curves at some
#   p strictly between them that was never checked (consistent with the
#   69.5% residual violation rate above, measured on a finer 8-value grid
#   than the 17-value grid used for training).
# - Single random seed per model (no repeated-run variance estimate) -- the
#   MSE/violation-rate numbers, while from a clean corrected single run, have
#   not been checked for robustness across multiple training seeds/splits.